In [ ]:
df_train = pd.read_csv("path/to/train.csv")
df_test = pd.read_csv("path/to/test.csv")

Separaçã das features do target

In [ ]:
target_col = "repeat_purchase_90d"
ignore_cols = ["ID", "customer_id", "purchase_date", target_col]

# Seleção automática de features preditivas (apenas numéricas e categóricas úteis)
feature_cols = [c for c in df_train.columns if c not in ignore_cols]

X = df_train[feature_cols].copy()
y = df_train[target_col].copy()

# Guardar os IDs do ficheiro de teste para o ficheiro final de submissão
test_ids = df_test_raw["ID"].copy()

# APLICAR NO TEST.CSV AS MESMAS TRANSFORMAÇÕES/FEATURES FEITAS NO TRAIN
# (Garantir que X_test tem exatamente as mesmas colunas que X)
X_test = df_test_raw[
    [c for c in feature_cols if c in df_test_raw.columns]
].copy()

treino com Stratified K-Fold (Cross-Validation)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(df_train))
test_preds = np.zeros(len(df_test_raw))

# Modelo baseline simples e robusto (LightGBM/HistGradientBoosting lida bem com NaNs e categóricas)
model = HistGradientBoostingClassifier(random_state=42)

auc_scores, f1_scores = [], []

print("=== INÍCIO DA CROSS-VALIDATION ===")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
  X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
  X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

  # Treinar o modelo
  model.fit(X_tr, y_tr)

  # Prever probabilidades no conjunto de validação
  val_probs = model.predict_proba(X_va)[:, 1]
  oof_preds[val_idx] = val_probs

  # Avaliar o fold
  fold_auc = roc_auc_score(y_va, val_probs)
  fold_f1 = f1_score(y_va, (val_probs >= 0.5).astype(int))

  auc_scores.append(fold_auc)
  f1_scores.append(fold_f1)

  print(
      f"Fold {fold+1} | ROC-AUC: {fold_auc:.4f} | F1-Score (threshold=0.5):"
      f" {fold_f1:.4f}"
  )

  # Somar previsões acumuladas para o dataset de teste
  test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

AVALIAÇÃO DE DESEMPENHO GLOBAL (OUT-OF-FOLD)

In [ ]:
overall_auc = roc_auc_score(y, oof_preds)
overall_logloss = log_loss(y, oof_preds)

print("\n=== DESEMPENHO GLOBAL DO MODELO BASELINE ===")
print(f"• Mean ROC-AUC (Cross-Validation): {np.mean(auc_scores):.4f}")
print(f"• Overall Out-of-Fold ROC-AUC:    {overall_auc:.4f}")
print(f"• Overall Out-of-Fold Log-Loss:   {overall_logloss:.4f}")


Submission file submission (sample_submission.csv format)

In [ ]:
submission = pd.DataFrame(
    {"ID": test_ids, "repeat_purchase_90d": test_preds}
)

submission_file = "/Users/renata/Library/CloudStorage/OneDrive-NOVAIMS/mestrado/1semestre/AprendizagemAutomatica(S1)/ProjectML/ML_Grupo18/data/submission_baseline.csv"
submission.to_csv(submission_file, index=False)

print(
    f"\nFicheiro de submissão gerado com sucesso em: {submission_file}"
)
print("Amostra das previsões:")
print(submission.head())